In [2]:
from pickle import FALSE
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from transformers import BertModel, BertTokenizer, BertForMaskedLM, BertForQuestionAnswering, T5ForConditionalGeneration, T5Tokenizer
from transformers import GPT2Tokenizer, GPT2LMHeadModel, GPT2Model
from transformers.models.bert.modeling_bert import BertEmbeddings, BertEncoder, BertPooler
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np
import time, os
from pynq import Overlay
from pynq import allocate
from l2_error import l2_error


WIDTH = 768
SMALL_WIDTH = 512
GROUP = 64
EPS = 1e-5

FPGA_cal_time = 0.0
CPU_cal_time = 0.0


class Connection:
    def __init__(self):
        self.overlay = Overlay('snake_v1_7.bit')
        self.acc_ip = self.overlay.accelerator_control_0
        self.buf0 = allocate((GROUP*WIDTH), dtype=np.uint16)
        self.buf1 = allocate((GROUP*WIDTH), dtype=np.uint16)
        self.out  = allocate((GROUP*WIDTH), dtype=np.uint16)
        self.pa0 = int(self.buf0.physical_address)
        self.pa1 = int(self.buf1.physical_address)
        self.pao = int(self.out.physical_address)
        self.wr(0x10,  self.pa0 & 0xFFFFFFFF)       # in0 low
        self.wr(0x14, (self.pa0 >> 32) & 0xFFFFFFFF)# in0 high
        self.wr(0x1C,  self.pa1 & 0xFFFFFFFF)       # in1 low
        self.wr(0x20, (self.pa1 >> 32) & 0xFFFFFFFF)# in1 high
        self.wr(0x28,  self.pao & 0xFFFFFFFF)       # out low
        self.wr(0x2C, (self.pao >> 32) & 0xFFFFFFFF)# out high
        print('初始化:')
        print("pa0 align:", self.pa0 % 8)
        print("pa1 align:", self.pa1 % 8)
        print("pao align:", self.pao % 8)
        print("in0 = 0x%08X_%08X"%(self.rd(0x14), self.rd(0x10)))
        print("in1 = 0x%08X_%08X"%(self.rd(0x20), self.rd(0x1C)))
        print("out = 0x%08X_%08X"%(self.rd(0x2C), self.rd(0x28)))
        print('\n')
    
    def wr(self, off, val): 
        self.acc_ip.write(off, int(val) & 0xFFFFFFFF)
        
    def rd(self, off): 
        return self.acc_ip.read(off)

    def bf16_to_f32(self, u16_arr: np.ndarray) -> np.ndarray:
        u32 = (u16_arr.astype(np.uint32) << 16)
        return u32.view(np.float32)
        
    def send_data(self, cfg, data1: Tensor, data2=None):
        arr0 = data1.to(torch.bfloat16).view(torch.uint16).numpy().flatten()
        np.copyto(self.buf0, arr0); 
        self.buf0.flush()
        
        if data2 is not None:
            arr1 = data2.to(torch.bfloat16).view(torch.uint16).numpy().flatten()
            np.copyto(self.buf1, arr1); 
            self.buf1.flush()
        self.out[:] = 0; 
        self.out.flush()
        
        #stage 0
        self.wr(0x34, 0)
        self.wr(0x00, 1)
        t0 = time.time()
        while (self.rd(0x00) & 0x2) == 0:
            if time.time() - t0 > 5.0:
                raise TimeoutError("stage 0 超时，检查 IP/时钟/复位")
            time.sleep(0.001)
        
        #stage 1
        self.wr(0x3C, cfg)
        self.wr(0x34, 1)
        self.wr(0x00, 1)
        t1 = time.perf_counter()
        while (self.rd(0x00) & 0x2) == 0:
            if time.perf_counter() - t1 > 10.0:
                raise TimeoutError(f"stage 1(config={cfg}) 超时")
        global FPGA_cal_time
        FPGA_cal_time += time.perf_counter() - t1
        
        #stage2
        self.wr(0x34, 2)   # stage = 2（搬运）
        self.wr(0x00, 1)
        t2 = time.time()
        while (self.rd(0x00) & 0x2) == 0:
            if time.time() - t2 > 5.0:
                raise TimeoutError(f"stage 2(config={cfg}) 超时")
            time.sleep(0.001)
        
        self.out.invalidate()
        out_vec = self.out.copy()  # 保存这一轮的输出
        
        result = self.bf16_to_f32(out_vec)
        return result

mul_nums = 0
add_nums = 0
gelu_nums = 0
layernorm_nums = 0
rmsnorm_nums = 0
con = Connection()

def calculate(type, data1, data2=None, L2_check=False, use_cpu=False):
    assert data1.dim() == 2 and data1.size(0) == GROUP and data1.size(1) == WIDTH, "Input shape must be (GROUP, WIDTH)"
    if data2 is not None:
        assert data2.dim() == 2 and data2.size(0) == GROUP and data2.size(1) == WIDTH, "Input shape must be (GROUP, WIDTH)"
    
    global CPU_cal_time
    t = time.perf_counter()
    if type=='gelu':
        #print("Calculating GELU for data shape:", data1.shape)
        if L2_check or use_cpu:
            ref = F.gelu(data1)
            CPU_cal_time += time.perf_counter() - t
        
        global gelu_nums
        gelu_nums += 1
        cfg = 4
    elif type=='mul':
        #print("Calculating MUL for data shapes:", data1.shape, data2.shape)
        if L2_check or use_cpu:
            ref = data1 * data2
            CPU_cal_time += time.perf_counter() - t
            
        global mul_nums
        mul_nums += 1
        cfg = 6
    elif type=='add':
        #print("Calculating ADD for data shapes:", data1.shape, data2.shape)
        if L2_check or use_cpu:
            ref = data1 + data2
            CPU_cal_time += time.perf_counter() - t
            
        global add_nums
        add_nums += 1
        cfg = 5
    elif type=='layernorm':
        #print("Calculating LAYERNORM for data shape:", data1.shape)
        if L2_check or use_cpu:
            mean = data1.mean(dim=-1, keepdim=True)
            var = data1.var(dim=-1, keepdim=True, unbiased=False)
            x_normalized = (data1 - mean) / torch.sqrt(var + EPS)
            ref = x_normalized
            CPU_cal_time += time.perf_counter() - t
            
        global layernorm_nums
        layernorm_nums += 1
        cfg = 1
    elif type=='rmsnorm':
        #print("Calculating RMSNORM for data shape:", data1.shape)
        if L2_check or use_cpu:
            variance = data1.to(torch.float32).pow(2).mean(-1, keepdim=True)
            data1 = data1 * torch.rsqrt(variance + EPS)
            ref = data1
            CPU_cal_time += time.perf_counter() - t
            
        global rmsnorm_nums
        rmsnorm_nums += 1
        cfg = 2
        
    if use_cpu:
        return ref
    else:
        result = con.send_data(cfg,data1,data2)
        
    if L2_check:
        dif_l2_error, points = l2_error(ref.flatten().numpy(), result)
        if dif_l2_error > 0.01:
            print(f"WARNING! config={type}, l2_error={dif_l2_error}, points = {points}")

    res_tensor = torch.from_numpy(result).view(GROUP, WIDTH)
    return res_tensor
    
def norm_and_cal(type, data1, data2=None):
    #print("Normalizing data1 shape:", data1.shape)
    assert data1.dim() == 2 and data1.size(1) == WIDTH, "Data1 shape must be (n, WIDTH)"
    assert data2 is None or (data2.dim() == 1 and data2.size(0) == WIDTH), "Data2 shape must be (1, WIDTH)"
    n = data1.size(0)
    rem = n % GROUP
    if rem == 0:
        pad = 0
        data1 = data1.view(n // GROUP, GROUP, WIDTH)
    else:
        pad = GROUP - rem
        pad_tensor = torch.zeros((pad, WIDTH), dtype=data1.dtype, device=data1.device)
        data1 = torch.cat([data1, pad_tensor], dim=0).view((n + pad) // GROUP, GROUP, WIDTH)
    
    result = torch.zeros_like(data1)
    
    if data2 is not None:
        data2 = data2.repeat(GROUP, 1)
        
    for i in range((n + pad) // GROUP):
        result[i,:,:] = calculate(type, data1[i,:,:], data2, L2_check=False, use_cpu=False)
    result = result.view(n + pad, WIDTH)[:n, :]
    return result       

class MyGELU(nn.Module):
    def __init__(self, is_decoder=False):
        super().__init__()
        self.is_decoder = is_decoder
    
    def forward(self, x: Tensor) -> Tensor:
        if gelu_nums == 0:
            print(f"Using custom {'decoder' if self.is_decoder else 'encoder'} GELU, shape:", x.shape)
        B, seq_len, dim = x.shape

        if dim % WIDTH != 0:
            pad = WIDTH - dim % WIDTH
            pad_tensor = torch.zeros((B, seq_len, pad), dtype=x.dtype)
            x = torch.cat([x, pad_tensor], dim=-1)
        else:
            pad = 0
        n = (dim + pad) // WIDTH
        x = x.view(B*seq_len*n, WIDTH)
        x = norm_and_cal('gelu', x)
        x = x.view(B, seq_len, dim+pad)[:, :, :dim]
        return x

class MyLayerNorm(nn.LayerNorm):
    def __init__(self, LayerNorm: nn.LayerNorm):
        super().__init__(LayerNorm.normalized_shape, LayerNorm.eps)
        #print("Creating custom LayerNorm, shape:", LayerNorm.normalized_shape)
    
    def forward(self, input: Tensor) -> Tensor:
        if layernorm_nums == 0:
            print("Using custom LayerNorm, input shape:", input.shape)
        B, seq_len, dim = input.shape
        for b in range(B):
            input[b,:,:] = norm_and_cal('layernorm', input[b,:,:])
            input[b,:,:] = norm_and_cal('mul', input[b,:,:], self.weight)
            if self.bias is not None:
                input[b,:,:] = norm_and_cal('add', input[b,:,:], self.bias)
        
        return input    
    
class MyLinear(nn.Linear):
    def __init__(self, Linear: nn.Linear):
        super().__init__(Linear.in_features, Linear.out_features, Linear.bias is not None)
        #print(f"Creating custom Linear layer: {Linear.in_features} -> {Linear.out_features}")
    
    def forward(self, input: Tensor) -> Tensor:
        if mul_nums == 0:
            print("Using custom Linear layer, input shape:", input.shape)
        B, seq_len, dim = input.shape
        out_dim, _ = self.weight.shape
        
        assert dim % WIDTH == 0, "Dimension must be multiple of WIDTH"
        n = dim // WIDTH
        input = input.view(B, seq_len, n, WIDTH)
        
        mat = torch.zeros((B, seq_len, out_dim), dtype=torch.float32)
        for b in range(B):
            for s in range(seq_len):
                for i in range(n):
                    res = norm_and_cal('mul', self.weight[:, i * WIDTH:(i + 1) * WIDTH], input[b, s, i, :])
                    mat[b, s, :] += res.sum(dim=-1)

        if self.bias is not None:
            if out_dim % WIDTH == 0:
                n = out_dim // WIDTH
                mat = mat.split(WIDTH, dim=2)
                bias_view = self.bias.view(n, WIDTH)
                for b in range(B):
                    for i in range(n):
                        mat[i][b, :, :] = norm_and_cal('add', mat[i][b, :, :], bias_view[i])
                mat = torch.cat(mat, dim=2)
            else:
                mat += self.bias
        return mat

class T5LayerNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        variance = hidden_states.to(torch.float32).pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)

        if self.weight.dtype in [torch.float16, torch.bfloat16]:
            hidden_states = hidden_states.to(self.weight.dtype)

        return self.weight * hidden_states

class MyRMSNorm(T5LayerNorm):
    def __init__(self, is_decoder: bool, LayerNorm: T5LayerNorm):
        #print("Creating custom RMSNorm, shape:", LayerNorm.weight.shape)
        super().__init__(LayerNorm.weight.shape, LayerNorm.variance_epsilon)
        self.is_decoder = is_decoder
        
    def forward(self, input: Tensor) -> Tensor:
        if rmsnorm_nums == 0:
            print(f"Using custom {'decoder' if self.is_decoder else 'encoder'} RMSNorm, input shape:", input.shape)
        B, seq_len, dim = input.shape
        pad = torch.zeros((B, seq_len, WIDTH - SMALL_WIDTH), dtype=torch.float32)
        input = torch.cat((input, pad), dim=2)
        delta = torch.sqrt(torch.full((WIDTH,), SMALL_WIDTH / WIDTH))
        pad = torch.zeros((WIDTH - SMALL_WIDTH), dtype=torch.float32)
        weight_pad = torch.cat((self.weight, pad), dim=0)
        delta = delta * weight_pad
        
        res = torch.empty((B, seq_len, SMALL_WIDTH), dtype=torch.float32)
        
        input = input.view(B*seq_len, WIDTH)
        input = norm_and_cal('rmsnorm', input)
        input = norm_and_cal('mul', input, delta)
        input = input.view(B, seq_len, WIDTH)
        res = input[:,:,:SMALL_WIDTH] 
        return res

class BoltT5ForConditionalGeneration(T5ForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        self.replace_activations()
        
    def replace_activations(self):
        #encoder
        self.encoder.final_layer_norm = MyRMSNorm(False, self.encoder.final_layer_norm)
        for block in self.encoder.block:
            for layer in block.layer:
                layer.layer_norm = MyRMSNorm(False, layer.layer_norm)
                if hasattr(layer, 'DenseReluDense'):
                    #print('this is T5LayerFF')
                    layer.DenseReluDense.act = MyGELU(False)
        #decoder
        self.decoder.final_layer_norm = MyRMSNorm(True, self.decoder.final_layer_norm)
        for block in self.decoder.block:
            for layer in block.layer:
                layer.layer_norm = MyRMSNorm(True, layer.layer_norm)
                if hasattr(layer, 'DenseReluDense'):
                    #print('this is T5LayerFF')
                    layer.DenseReluDense.act = MyGELU(True)    
   
class BoltBertModel(BertModel):
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self._replace_activations()
    
    def _replace_activations(self):
        for layer in self.encoder.layer:
            #BertLayer
            ##BertAttention
            ###BertSelfAttention
            layer.attention.self.query = MyLinear(layer.attention.self.query)
            layer.attention.self.key = MyLinear(layer.attention.self.key)
            layer.attention.self.value = MyLinear(layer.attention.self.value)
            ###BertSelfOutput
            layer.attention.output.dense = MyLinear(layer.attention.output.dense)
            layer.attention.output.LayerNorm = MyLayerNorm(layer.attention.output.LayerNorm)
            ##BertIntermediate
            layer.intermediate.dense = MyLinear(layer.intermediate.dense)
            layer.intermediate.intermediate_act_fn = MyGELU()
            ##BertOutput
            layer.output.dense = MyLinear(layer.output.dense)
            layer.output.LayerNorm = MyLayerNorm(layer.output.LayerNorm)

class BoltBertForMaskedLM(BertForMaskedLM):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BoltBertModel(config)
        
class BoltBertForQuestionAnswering(BertForQuestionAnswering):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BoltBertModel(config)

class BoltGPT2Model(GPT2Model):
    def __init__(self, config):
        super().__init__(config)
        self.replace_activations()
    
    def replace_activations(self):
        self.ln_f = MyLayerNorm(self.ln_f)
        for block in self.h:
            #block.ln_1 = MyLayerNorm(block.ln_1)
            #block.ln_2 = MyLayerNorm(block.ln_2)
            block.mlp.act = MyGELU()

class BoltGPT2LMHeadModel(GPT2LMHeadModel):
    def __init__(self, config):
        super().__init__(config)
        self.transformer = BoltGPT2Model(config)

def bert_text_generation(text_with_mask, top_k=5, layers_num = 12):
    # 创建模型和分词器
    model = BoltBertForMaskedLM.from_pretrained('rbt'+str(layers_num))
    tokenizer = BertTokenizer.from_pretrained('rbt'+str(layers_num))
    model.eval() # 设置为评估模式
    
    inputs = tokenizer(text_with_mask, return_tensors="pt")
    with torch.no_grad():
        outputs1 = model(**inputs)
        predictions = outputs1.logits
    
    mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    # 获取预测结果
    mask_logits = predictions[0, mask_token_index, :]
    top_tokens = torch.topk(mask_logits, top_k, dim=1).indices[0].tolist()
    # 解码结果
    generated_texts = []
    for token in top_tokens:
        generated = text_with_mask.replace(tokenizer.mask_token, tokenizer.decode([token]))
        generated_texts.append(generated)
    print('\n')
    print(f'原文: {text_with_mask}')
    for i, text in enumerate(generated_texts):
        print(f"选项 {i+1}: {text}")

def t5_translation(target='en'):
    model = BoltT5ForConditionalGeneration.from_pretrained('./t5')
    tokenizer = T5Tokenizer.from_pretrained('./t5')
    src_text = '你好,我是一个t5语言模型'
    
    prefix = 'translate to '+target+': '
    src_text = prefix + src_text

    input_ids = tokenizer(src_text, return_tensors="pt")
    generated_tokens = model.generate(**input_ids)
    result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    print('\n')
    print(f'原文：{src_text}')
    print(f'翻译结果：{result[0]}')
    print('\n')

def GPT2_text_generation():
    tokenizer = GPT2Tokenizer.from_pretrained('./GPT2')
    model = BoltGPT2LMHeadModel.from_pretrained('./GPT2')
    question = "When I first met him"

    input_ids = tokenizer.encode(question, return_tensors='pt')
    output = model.generate(
        input_ids,
        max_length=30,            # 生成文本的最大长度
        num_return_sequences=1,   # 生成几个候选结果
        temperature=0.05,         # 控制随机性：较低值更确定，较高值更多样
        repetition_penalty=1.2,   # 重复惩罚：大于1.0的值会降低重复词的概率
        do_sample=True            # 启用采样方法；如果为False，则使用贪心搜索
    )
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print('\n')
    print(f'原文：{question}')
    print(f'续写：{generated_text}')
    print('\n')
if __name__ == "__main__":
    total_time = time.perf_counter()
    print('\nbert文本填空任务:')
    bert_text_generation("我是一只[MASK]。", top_k=5, layers_num=3)
    print(f"gelu_nums: {gelu_nums}, mul_nums: {mul_nums}, add_nums: {add_nums}, layernorm_nums: {layernorm_nums}, rmsnorm_nums: {rmsnorm_nums}")
    print(f"total_time: {time.perf_counter()-total_time}")
    print(f"FPGA_cal_time: {FPGA_cal_time}")
    gelu_nums = 0
    mul_nums = 0
    add_nums = 0
    layernorm_nums = 0
    rmsnorm_nums = 0
    print('\nt5模型中英文翻译任务:')
    t5_translation('en')
    print(f"gelu_nums: {gelu_nums}, mul_nums: {mul_nums}, add_nums: {add_nums}, layernorm_nums: {layernorm_nums}, rmsnorm_nums: {rmsnorm_nums}")
    print(f"total_time: {time.perf_counter()-total_time}")
    print(f"FPGA_cal_time: {FPGA_cal_time}")
    gelu_nums = 0
    mul_nums = 0
    add_nums = 0
    layernorm_nums = 0
    rmsnorm_nums = 0
    print('\nGPT2文本生成任务:')
    GPT2_text_generation()
    print(f"gelu_nums: {gelu_nums}, mul_nums: {mul_nums}, add_nums: {add_nums}, layernorm_nums: {layernorm_nums}, rmsnorm_nums: {rmsnorm_nums}")
    print(f"total_time: {time.perf_counter()-total_time}")
    print(f"FPGA_cal_time: {FPGA_cal_time}")
    

初始化:
pa0 align: 0
pa1 align: 0
pao align: 0
in0 = 0x00000000_37CA0000
in1 = 0x00000000_37CC0000
out = 0x00000000_37CE0000



bert文本填空任务:
Using custom Linear layer, input shape: torch.Size([1, 8, 768])
Using custom LayerNorm, input shape: torch.Size([1, 8, 768])
Using custom encoder GELU, shape: torch.Size([1, 8, 3072])


原文: 我是一只[MASK]。
选项 1: 我是一只鹿。
选项 2: 我是一只物。
选项 3: 我是一只鼠。
选项 4: 我是一只猫。
选项 5: 我是一只鸟。
gelu_nums: 3, mul_nums: 3462, add_nums: 33, layernorm_nums: 6, rmsnorm_nums: 0
total_time: 19.793171433000225
FPGA_cal_time: 0.041941758987832145

t5模型中英文翻译任务:
Using custom encoder RMSNorm, input shape: torch.Size([1, 14, 512])
Using custom encoder GELU, shape: torch.Size([1, 14, 1024])


原文：translate to en: 你好,我是一个t5语言模型
翻译结果：Hello, I'm a T5 language model


gelu_nums: 104, mul_nums: 317, add_nums: 0, layernorm_nums: 0, rmsnorm_nums: 317
total_time: 35.144281138000224
FPGA_cal_time: 0.05091646098162528

GPT2文本生成任务:


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Using custom encoder GELU, shape: torch.Size([1, 5, 3072])
Using custom LayerNorm, input shape: torch.Size([1, 5, 768])


原文：When I first met him
续写：When I first met him in the early 1990s, he was a very nice guy. He had an amazing personality and his attitude is that you can


gelu_nums: 150, mul_nums: 25, add_nums: 25, layernorm_nums: 25, rmsnorm_nums: 0
total_time: 45.749680178999824
FPGA_cal_time: 0.05372215798252
